# 20 · Train-on-GUE, Predict-Zeta

This notebook pushes the transfer idea one step further.

Earlier notebooks showed:
- zeta vs Poisson is easy
- GUE vs Poisson is easy
- zeta vs GUE is hard
- these patterns transfer across height

Here we ask a more pointed question:

> If we learn a decision boundary from **GUE vs Poisson**, does it classify **zeta vs Poisson** the same way?

And conversely:

> If we learn from **zeta vs Poisson**, does it classify **GUE vs Poisson** the same way?

## Why this matters

If a classifier trained on GUE can predict zeta against Poisson, that supports the claim that the learned low-dimensional boundary is not merely similar by eye — it is operationally transferable between the two ensembles.

In [ ]:
import numpy as np
import mpmath as mp
import matplotlib.pyplot as plt

mp.mp.dps = 50
rng = np.random.default_rng(9423)

## Base data

In [ ]:
N = 1800
zeros = [mp.zetazero(n) for n in range(1, N + 1)]
t = np.array([float(mp.im(z)) for z in zeros])
zeta_gaps_full = np.diff(t)

poisson_gaps_full = rng.exponential(scale=1.0, size=len(zeta_gaps_full))

def sample_gue_bulk_spacings(matrix_size=150, n_matrices=70, bulk_fraction=0.5, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    all_gaps = []
    for _ in range(n_matrices):
        real = rng.normal(size=(matrix_size, matrix_size))
        imag = rng.normal(size=(matrix_size, matrix_size))
        A = real + 1j * imag
        H = (A + A.conj().T) / 2.0
        eigvals = np.linalg.eigvalsh(H)
        eigvals = np.sort(eigvals)

        n = len(eigvals)
        keep = int(n * bulk_fraction)
        start = (n - keep) // 2
        stop = start + keep
        bulk_vals = eigvals[start:stop]
        bulk_gaps = np.diff(bulk_vals)
        all_gaps.extend(bulk_gaps.tolist())

    return np.array(all_gaps)

gue_gaps_full = sample_gue_bulk_spacings(matrix_size=150, n_matrices=70, bulk_fraction=0.5, rng=rng)

len(zeta_gaps_full), len(poisson_gaps_full), len(gue_gaps_full)

## Minimal feature set

We again use the compact 3-feature set:
- entropy
- unevenness
- ratio_mean

In [ ]:
def extract_windows(gaps, k=5):
    return np.array([gaps[i:i+k] for i in range(len(gaps) - k + 1)])

def normalize_window(window):
    w = np.array(window, dtype=float)
    return w / w.mean()

def unevenness(window):
    return float(np.max(window) - np.min(window))

def window_entropy(window):
    eps = 1e-12
    p = window / np.sum(window)
    return float(-np.sum(p * np.log(p + eps)))

def ratio_mean(window):
    g1 = window[:-1]
    g2 = window[1:]
    r = np.minimum(g1, g2) / np.maximum(g1, g2)
    return float(np.mean(r))

def build_feature_matrix(gaps, k=5):
    windows = extract_windows(gaps, k=k)
    windows_n = np.array([normalize_window(w) for w in windows])
    X = np.array([
        [window_entropy(w), unevenness(w), ratio_mean(w)]
        for w in windows_n
    ], dtype=float)
    return windows_n, X

## Classifier helpers

In [ ]:
def standardize_train_test(X_train, X_test):
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0)
    std[std == 0] = 1.0
    return (X_train - mean) / std, (X_test - mean) / std

def nearest_centroid_fit(X_train, y_train):
    classes = np.unique(y_train)
    centroids = {c: X_train[y_train == c].mean(axis=0) for c in classes}
    return centroids

def nearest_centroid_predict(X, centroids):
    classes = sorted(centroids.keys())
    centroid_array = np.vstack([centroids[c] for c in classes])
    dists = np.linalg.norm(X[:, None, :] - centroid_array[None, :, :], axis=2)
    pred_idx = np.argmin(dists, axis=1)
    return np.array([classes[i] for i in pred_idx])

def knn_predict(X_train, y_train, X_test, k=5):
    preds = []
    for x in X_test:
        d = np.linalg.norm(X_train - x, axis=1)
        nn = np.argsort(d)[:k]
        votes = y_train[nn]
        counts = np.bincount(votes, minlength=2)
        preds.append(np.argmax(counts))
    return np.array(preds)

def accuracy(y_true, y_pred):
    return float(np.mean(y_true == y_pred))

def confusion_2x2(y_true, y_pred):
    return np.array([
        [np.sum((y_true == 0) & (y_pred == 0)), np.sum((y_true == 0) & (y_pred == 1))],
        [np.sum((y_true == 1) & (y_pred == 0)), np.sum((y_true == 1) & (y_pred == 1))]
    ])

## Prepare one train block and several test blocks

We keep the same block structure as the previous notebook.

In [ ]:
block_size = 300
block_starts = [0, 300, 600, 900, 1200]
block_labels = [f"{s+1}-{s+block_size}" for s in block_starts]

zeta_blocks = {}
poisson_blocks = {}

for s, label in zip(block_starts, block_labels):
    zeta_block = zeta_gaps_full[s:s + block_size]
    poisson_block = poisson_gaps_full[s:s + block_size]
    _, zeta_X = build_feature_matrix(zeta_block, k=5)
    _, poisson_X = build_feature_matrix(poisson_block, k=5)
    zeta_blocks[label] = zeta_X
    poisson_blocks[label] = poisson_X

_, gue_X_full = build_feature_matrix(gue_gaps_full[:1400], k=5)

{label: zeta_blocks[label].shape for label in block_labels}, gue_X_full.shape

## Train on GUE vs Poisson, test on zeta vs Poisson

Class 0 = structured ensemble  
Class 1 = Poisson

In [ ]:
def run_cross_ensemble_transfer(train_structured, train_poisson, test_structured, test_poisson):
    m_train = min(len(train_structured), len(train_poisson))
    m_test = min(len(test_structured), len(test_poisson))

    X_train = np.vstack([train_structured[:m_train], train_poisson[:m_train]])
    y_train = np.array([0] * m_train + [1] * m_train)

    X_test = np.vstack([test_structured[:m_test], test_poisson[:m_test]])
    y_test = np.array([0] * m_test + [1] * m_test)

    X_train_s, X_test_s = standardize_train_test(X_train, X_test)

    centroids = nearest_centroid_fit(X_train_s, y_train)
    y_pred_centroid = nearest_centroid_predict(X_test_s, centroids)
    y_pred_knn = knn_predict(X_train_s, y_train, X_test_s, k=5)

    return {
        "centroid_accuracy": accuracy(y_test, y_pred_centroid),
        "knn_accuracy": accuracy(y_test, y_pred_knn),
        "centroid_confusion": confusion_2x2(y_test, y_pred_centroid).tolist(),
        "knn_confusion": confusion_2x2(y_test, y_pred_knn).tolist(),
    }

In [ ]:
train_label = block_labels[0]
test_labels = block_labels[1:]

gue_to_zeta_results = []
zeta_to_gue_results = []

for test_label in test_labels:
    train_poisson = poisson_blocks[train_label]
    test_poisson = poisson_blocks[test_label]

    zeta_test = zeta_blocks[test_label]
    zeta_train = zeta_blocks[train_label]

    m_train_gue = min(len(gue_X_full), len(train_poisson), len(zeta_train))
    m_test_gue = min(len(gue_X_full), len(test_poisson), len(zeta_test))

    gue_train = gue_X_full[:m_train_gue]
    gue_test = gue_X_full[-m_test_gue:]

    gue_to_zeta_results.append({
        "train_block": train_label,
        "test_block": test_label,
        "direction": "train GUE vs Poisson → test zeta vs Poisson",
        "metrics": run_cross_ensemble_transfer(
            gue_train, train_poisson,
            zeta_test, test_poisson
        ),
    })

    zeta_to_gue_results.append({
        "train_block": train_label,
        "test_block": test_label,
        "direction": "train zeta vs Poisson → test GUE vs Poisson",
        "metrics": run_cross_ensemble_transfer(
            zeta_train, train_poisson,
            gue_test, test_poisson
        ),
    })

gue_to_zeta_results[:1], zeta_to_gue_results[:1]

## Accuracy comparison

In [ ]:
x = np.arange(len(test_labels))
width = 0.18

g2z_centroid = [r["metrics"]["centroid_accuracy"] for r in gue_to_zeta_results]
g2z_knn = [r["metrics"]["knn_accuracy"] for r in gue_to_zeta_results]

z2g_centroid = [r["metrics"]["centroid_accuracy"] for r in zeta_to_gue_results]
z2g_knn = [r["metrics"]["knn_accuracy"] for r in zeta_to_gue_results]

fig, axes = plt.subplots(1, 2, figsize=(12.8, 4.8), sharey=True)

axes[0].bar(x - width/2, g2z_centroid, width, label="nearest centroid")
axes[0].bar(x + width/2, g2z_knn, width, label="k-NN (k=5)")
axes[0].set_xticks(x, test_labels, rotation=25)
axes[0].set_ylim(0.4, 1.0)
axes[0].set_title("train GUE vs Poisson → test zeta vs Poisson")
axes[0].set_ylabel("accuracy")
axes[0].legend()

axes[1].bar(x - width/2, z2g_centroid, width, label="nearest centroid")
axes[1].bar(x + width/2, z2g_knn, width, label="k-NN (k=5)")
axes[1].set_xticks(x, test_labels, rotation=25)
axes[1].set_ylim(0.4, 1.0)
axes[1].set_title("train zeta vs Poisson → test GUE vs Poisson")
axes[1].set_ylabel("accuracy")

plt.tight_layout()
plt.show()

## Compare against same-ensemble transfer baselines

For context, we also compute:
- train zeta vs Poisson → test zeta vs Poisson
- train GUE vs Poisson → test GUE vs Poisson

In [ ]:
baseline_results = []

for test_label in test_labels:
    train_poisson = poisson_blocks[train_label]
    test_poisson = poisson_blocks[test_label]

    zeta_train = zeta_blocks[train_label]
    zeta_test = zeta_blocks[test_label]

    m_train_gue = min(len(gue_X_full), len(train_poisson), len(zeta_train))
    m_test_gue = min(len(gue_X_full), len(test_poisson), len(zeta_test))

    gue_train = gue_X_full[:m_train_gue]
    gue_test = gue_X_full[-m_test_gue:]

    baseline_results.append({
        "test_block": test_label,
        "zeta_to_zeta": run_cross_ensemble_transfer(
            zeta_train, train_poisson,
            zeta_test, test_poisson
        ),
        "gue_to_gue": run_cross_ensemble_transfer(
            gue_train, train_poisson,
            gue_test, test_poisson
        ),
    })

baseline_results[:1]

In [ ]:
x = np.arange(len(test_labels))
width = 0.18

zeta_base = [r["zeta_to_zeta"]["knn_accuracy"] for r in baseline_results]
gue_base = [r["gue_to_gue"]["knn_accuracy"] for r in baseline_results]
g2z = [r["metrics"]["knn_accuracy"] for r in gue_to_zeta_results]
z2g = [r["metrics"]["knn_accuracy"] for r in zeta_to_gue_results]

plt.figure(figsize=(10.5, 4.8))
plt.bar(x - 1.5*width, zeta_base, width, label="zeta→zeta baseline")
plt.bar(x - 0.5*width, gue_base, width, label="GUE→GUE baseline")
plt.bar(x + 0.5*width, g2z, width, label="GUE→zeta")
plt.bar(x + 1.5*width, z2g, width, label="zeta→GUE")
plt.xticks(x, test_labels, rotation=25)
plt.ylim(0.4, 1.0)
plt.ylabel("k-NN accuracy")
plt.title("Same-ensemble vs cross-ensemble transfer")
plt.legend()
plt.tight_layout()
plt.show()

## Confusion matrices

In [ ]:
for label, records in [("GUE→zeta", gue_to_zeta_results), ("zeta→GUE", zeta_to_gue_results)]:
    print("\n" + "="*72)
    print(label)
    for r in records:
        print(f"\ntrain {r['train_block']} → test {r['test_block']}")
        print("Nearest-centroid confusion:")
        print(np.array(r["metrics"]["centroid_confusion"]))
        print("k-NN confusion:")
        print(np.array(r["metrics"]["knn_confusion"]))

## Compact summary table

In [ ]:
summary = {
    "gue_to_zeta": [
        {
            "train_block": r["train_block"],
            "test_block": r["test_block"],
            "centroid_accuracy": r["metrics"]["centroid_accuracy"],
            "knn_accuracy": r["metrics"]["knn_accuracy"],
        }
        for r in gue_to_zeta_results
    ],
    "zeta_to_gue": [
        {
            "train_block": r["train_block"],
            "test_block": r["test_block"],
            "centroid_accuracy": r["metrics"]["centroid_accuracy"],
            "knn_accuracy": r["metrics"]["knn_accuracy"],
        }
        for r in zeta_to_gue_results
    ],
    "baselines": [
        {
            "test_block": r["test_block"],
            "zeta_to_zeta_knn": r["zeta_to_zeta"]["knn_accuracy"],
            "gue_to_gue_knn": r["gue_to_gue"]["knn_accuracy"],
        }
        for r in baseline_results
    ],
}
summary

## Notes

- If training on GUE vs Poisson successfully predicts zeta vs Poisson, that supports the idea that the boundary learned from GUE is operationally the same as the one needed for zeta.
- If the reciprocal transfer also works, that strengthens the interpretation that zeta and GUE share a common low-dimensional separator from Poisson.
- This is still a simple classifier test, not a proof of universality.

## Next directions

A natural next notebook could do one of these:

1. add bootstrap uncertainty on cross-ensemble transfer accuracies  
2. test several alternative minimal feature sets  
3. compare train-on-middle, test-on-both-ends transfer  
4. extend the same idea to conditional windows